In [ ]:
import json
import boto3

In [2]:
with open("eval_input.json", "r") as f:
    input_data = json.load(f)
input_data

{'paper_id': '9aff0461b7e1ecceef4f4b97027a83bda7163dbe',
 'metadata': {'title': 'Effects of DASH diet with or without time-restricted eating in the management of stage 1 primary hypertension: a randomized controlled trial',
  'year': 2024,
  'venue': 'Nutrix Journal',
  'field_of_study': ['Medicine'],
  'publication_type': ['JournalArticle']},
 'claims': [{'claim': 'The reduction of systolic blood pressure was 5.595 ± 4.072 mm Hg in the DASH group and 8.459 ± 4.260 mm Hg in the DASH + TRE group',
   'evidence': ['The reduction of systolic blood pressure and diastolic blood pressure were 5.595 ± 4.072 and 5.351 ± 5.643 mm Hg in the DASH group and 8.459 ± 4.260 and 9.459 ± 4.375 mm Hg in the DASH + TRE group.'],
   'sections': ['Abstract - Results'],
   'strength': 'strong',
   'source_chunks': ['9aff0461b7e1ecceef4f4b97027a83bda7163dbe_chunk_000']},
  {'claim': 'The reduction of diastolic blood pressure was 5.351 ± 5.643 mm Hg in the DASH group and 9.459 ± 4.375 mm Hg in the DASH + TRE 

In [9]:
def build_user_prompt(data: dict) -> str:
    return f"""
You are a scientific study quality assessment agent.

Your role is to evaluate the reliability of a research study based strictly on the provided claims and metadata.

Data:
\"\"\"
{data}
\"\"\"

Guidelines:
- Do not infer methods or details that are not explicitly supported by the provided information.
- If any information is missing or unclear, explicitly mark it as "unclear."
- Be conservative in your assessment and avoid overconfidence in your conclusions.

Output:
- Return ONLY a valid JSON object that adheres to the following schema:
{{
  "paper_id": "...",
  "study_quality": {{
    "study_design": "RCT | observational | cohort | meta-analysis | unclear",
    "sample_size": "large | medium | small | unclear",
    "bias_risk": "low | moderate | high",
    "evidence_strength": "strong | moderate | weak",
    "notes": "short justification grounded in claims"
  }}
}}

Strict Rules:
- Do NOT include any additional text, explanations, or comments outside the JSON object.
- Ensure the JSON is well-formed and adheres to the schema exactly.
""".strip()

In [10]:
conversation = [
    {
        "role": "user",
        "content": [{"text":build_user_prompt(input_data)}]
    }
]


In [6]:
brt = boto3.client("bedrock-runtime")

In [12]:
model_id= "global.anthropic.claude-haiku-4-5-20251001-v1:0"
response = brt.converse(
    modelId= model_id,
    messages=conversation

)

In [20]:
raw_output = response["output"]["message"]["content"][0]["text"]

In [15]:
print(response["output"]["message"]["content"][0]["text"])

```json
{
  "paper_id": "9aff0461b7e1ecceef4f4b97027a83bda7163dbe",
  "study_quality": {
    "study_design": "RCT",
    "sample_size": "small",
    "bias_risk": "moderate",
    "evidence_strength": "strong",
    "notes": "Study is explicitly described as a randomized controlled trial with two intervention groups (DASH and DASH+TRE). Sample sizes appear modest (n=26 for DASH group, n=37 for DASH+TRE group, total n=63), classified as small. Evidence strength is strong based on specific numerical outcomes with standard deviations and statistical significance testing reported. Moderate bias risk due to: limited sample size, unclear blinding status, no mention of allocation concealment or loss to follow-up, and publication venue is 'Nutrix Journal' which lacks established reputation metrics in provided data."
  }
}
```


## validation (forcing enum values, fail closed - downgrade confidence if something is invalid or missing)

In [16]:
import json
def clean_json(json_text:str) -> list:
    cleaned_json = json_text.strip()
    cleaned_json = cleaned_json.removeprefix("```json").removesuffix("```").strip()
    try:
        return json.loads(cleaned_json)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        return []


In [17]:
ALLOWED_STUDY_DESIGN = {
    "rct", "observational", "cohort", "meta-analysis", "unclear"
}

ALLOWED_SAMPLE_SIZE = {
    "large", "medium", "small", "unclear"
}

ALLOWED_BIAS_RISK = {
    "low", "moderate", "high"
}

ALLOWED_EVIDENCE_STRENGTH = {
    "strong", "moderate", "weak"
}

In [18]:
def normalize_enum(value, allowed, default="unclear"):
    if not value:
        return default
    value = value.strip().lower()
    return value if value in allowed else default

In [19]:
def validate_study_quality(raw_output: dict) -> dict:
    sq = raw_output.get("study_quality", {})

    validated = {
        "study_design": normalize_enum(
            sq.get("study_design"),
            ALLOWED_STUDY_DESIGN
        ),
        "sample_size": normalize_enum(
            sq.get("sample_size"),
            ALLOWED_SAMPLE_SIZE
        ),
        "bias_risk": normalize_enum(
            sq.get("bias_risk"),
            ALLOWED_BIAS_RISK,
            default="high"   # conservative default
        ),
        "evidence_strength": normalize_enum(
            sq.get("evidence_strength"),
            ALLOWED_EVIDENCE_STRENGTH,
            default="weak"
        ),
        "notes": sq.get("notes", "").strip()[:500]
    }

    return {
        "paper_id": raw_output.get("paper_id"),
        "study_quality": validated
    }

In [21]:
json_output = clean_json(raw_output)
json_output

{'paper_id': '9aff0461b7e1ecceef4f4b97027a83bda7163dbe',
 'study_quality': {'study_design': 'RCT',
  'sample_size': 'small',
  'bias_risk': 'moderate',
  'evidence_strength': 'strong',
  'notes': "Study is explicitly described as a randomized controlled trial with two intervention groups (DASH and DASH+TRE). Sample sizes appear modest (n=26 for DASH group, n=37 for DASH+TRE group, total n=63), classified as small. Evidence strength is strong based on specific numerical outcomes with standard deviations and statistical significance testing reported. Moderate bias risk due to: limited sample size, unclear blinding status, no mention of allocation concealment or loss to follow-up, and publication venue is 'Nutrix Journal' which lacks established reputation metrics in provided data."}}

In [22]:
validated_output = validate_study_quality(json_output)
validated_output

{'paper_id': '9aff0461b7e1ecceef4f4b97027a83bda7163dbe',
 'study_quality': {'study_design': 'rct',
  'sample_size': 'small',
  'bias_risk': 'moderate',
  'evidence_strength': 'strong',
  'notes': 'Study is explicitly described as a randomized controlled trial with two intervention groups (DASH and DASH+TRE). Sample sizes appear modest (n=26 for DASH group, n=37 for DASH+TRE group, total n=63), classified as small. Evidence strength is strong based on specific numerical outcomes with standard deviations and statistical significance testing reported. Moderate bias risk due to: limited sample size, unclear blinding status, no mention of allocation concealment or loss to follow-up, and public'}}

In [23]:
def attach_quality_to_claims(
    claims: list[dict],
    study_quality_result: dict,
) -> list[dict]:
    """
    Attaches paper-level study quality metadata to each claim.

    Args:
        claims (list[dict]): Claims extracted from a single paper
        study_quality_result (dict): Output of study quality agent

    Returns:
        list[dict]: Claims enriched with study_quality
    """

    if not claims:
        return []

    # Fail-closed default quality
    default_quality = {
        "study_design": "unclear",
        "sample_size": "unclear",
        "bias_risk": "high",
        "evidence_strength": "weak",
        "notes": ""
    }

    quality = study_quality_result.get("study_quality", default_quality)

    enriched_claims = []

    for claim in claims:
        enriched_claim = {
            **claim,
            "study_quality": quality
        }
        enriched_claims.append(enriched_claim)

    return enriched_claims

In [ ]:
evaluator = QualityEvaluator()
evaluated_papers = []
for paper in all_papers_claims:
    evaluated = evaluator.evaluate_study_quality(paper)
    validated = validate_study_quality(evaluated)
    enriched_claims = attach_quality_to_claims(paper["claims"], validated)
    updated_claims = [{**claim, "paper_id":paper['paper_id'], "title":paper['metadata']['title']} for claim in enriched_claims]
    evaluated_papers.append(updated_claims)
    

In [24]:
enriched_claims = attach_quality_to_claims(input_data["claims"], validated_output)

In [ ]:
enriched_claims
# looping, add paper_id and title

[{'claim': 'The reduction of systolic blood pressure was 5.595 ± 4.072 mm Hg in the DASH group and 8.459 ± 4.260 mm Hg in the DASH + TRE group',
  'evidence': ['The reduction of systolic blood pressure and diastolic blood pressure were 5.595 ± 4.072 and 5.351 ± 5.643 mm Hg in the DASH group and 8.459 ± 4.260 and 9.459 ± 4.375 mm Hg in the DASH + TRE group.'],
  'sections': ['Abstract - Results'],
  'strength': 'strong',
  'source_chunks': ['9aff0461b7e1ecceef4f4b97027a83bda7163dbe_chunk_000'],
  'study_quality': {'study_design': 'rct',
   'sample_size': 'small',
   'bias_risk': 'moderate',
   'evidence_strength': 'strong',
   'notes': 'Study is explicitly described as a randomized controlled trial with two intervention groups (DASH and DASH+TRE). Sample sizes appear modest (n=26 for DASH group, n=37 for DASH+TRE group, total n=63), classified as small. Evidence strength is strong based on specific numerical outcomes with standard deviations and statistical significance testing reported

In [27]:
paper_id = input_data["paper_id"]
paper_title = input_data["metadata"]["title"]

#for claim in enriched_claims:
#    claim.update({"paper_id":paper_id, "title":paper_title})

updated_claims = [{**claim, "paper_id":paper_id, "title":paper_title} for claim in enriched_claims]
updated_claims

[{'claim': 'The reduction of systolic blood pressure was 5.595 ± 4.072 mm Hg in the DASH group and 8.459 ± 4.260 mm Hg in the DASH + TRE group',
  'evidence': ['The reduction of systolic blood pressure and diastolic blood pressure were 5.595 ± 4.072 and 5.351 ± 5.643 mm Hg in the DASH group and 8.459 ± 4.260 and 9.459 ± 4.375 mm Hg in the DASH + TRE group.'],
  'sections': ['Abstract - Results'],
  'strength': 'strong',
  'source_chunks': ['9aff0461b7e1ecceef4f4b97027a83bda7163dbe_chunk_000'],
  'study_quality': {'study_design': 'rct',
   'sample_size': 'small',
   'bias_risk': 'moderate',
   'evidence_strength': 'strong',
   'notes': 'Study is explicitly described as a randomized controlled trial with two intervention groups (DASH and DASH+TRE). Sample sizes appear modest (n=26 for DASH group, n=37 for DASH+TRE group, total n=63), classified as small. Evidence strength is strong based on specific numerical outcomes with standard deviations and statistical significance testing reported

In [ ]:
{
  "id": "<paper_id>::<claim_index>",
  "embedding_text": "Claim: ... Evidence: ...",
  "metadata": {
    "paper_id": "...",
    "claim": "...",
    "evidence": "...",
    "section": "...",
    "study_quality": {...},
    "title": "...",
    "year": 2022
  }
}

In [28]:
with open("data_to_embed.json", "w") as f:
    json.dump(updated_claims, f)